# Sales Analysis - School Supplies Store

Demo project to practice **Git** commands.

Every time we modify this notebook, we'll make commits, branches, and merges to see Git in action.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('sales_data.csv', parse_dates=['date'])
df.head()

## 1. Quick data overview

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

## 2. Overall summary of units sold

In [ ]:
summary = df.groupby('product')['units_sold'].sum().sort_values(ascending=False)
summary

In [ ]:
summary.plot(kind='bar', title='Units sold by product', color='steelblue')
plt.ylabel('Units')
plt.show()

## 3. Revenue calculation

In [ ]:
df['revenue'] = df['units_sold'] * df['unit_price']
revenue_by_product = df.groupby('product')['revenue'].sum().sort_values(ascending=False)
revenue_by_product

In [ ]:
revenue_by_product.plot(kind='bar', title='Total revenue by product', color='darkorange')
plt.ylabel('Revenue ($)')
plt.show()

In [ ]:
revenue_by_product.plot(kind='pie', title='Revenue share by product', autopct='%1.1f%%', ylabel='')
plt.show()

## 4. Monthly sales trend

In [ ]:
monthly = df.set_index('date').resample('ME')['units_sold'].sum()
monthly

In [ ]:
monthly.plot(kind='line', marker='o', title='Monthly units sold (all products)')
plt.ylabel('Units')
plt.xlabel('Month')
plt.show()

## 5. 7-day rolling average (smoothing daily noise)

In [ ]:
daily_total = df.groupby('date')['units_sold'].sum()
rolling_avg = daily_total.rolling(window=7).mean()

plt.figure(figsize=(10,4))
plt.plot(daily_total.index, daily_total.values, alpha=0.3, label='Daily total')
plt.plot(rolling_avg.index, rolling_avg.values, color='crimson', label='7-day rolling avg')
plt.legend()
plt.title('Daily units sold vs. 7-day rolling average')
plt.show()

## 6. Pivot table: units sold by product and month

In [ ]:
df['month'] = df['date'].dt.to_period('M').astype(str)
pivot = df.pivot_table(index='month', columns='product', values='units_sold', aggfunc='sum')
pivot

In [ ]:
pivot.plot(kind='line', figsize=(10,5), title='Units sold by product over time')
plt.ylabel('Units')
plt.show()

## 7. Filtering and sorting

In [ ]:
# Days where Backpacks sold more than 20 units
high_backpack_days = df[(df['product'] == 'Backpacks') & (df['units_sold'] > 20)]
high_backpack_days.sort_values('units_sold', ascending=False).head(10)

## 8. Distribution of units sold per product (box plot)

In [ ]:
df.boxplot(column='units_sold', by='product', figsize=(8,5))
plt.title('Distribution of units sold by product')
plt.suptitle('')
plt.ylabel('Units sold')
plt.show()

## 9. Correlation between units sold and revenue

In [ ]:
df[['units_sold', 'unit_price', 'revenue']].corr()

## 10. Multiple aggregations at once

In [ ]:
agg_summary = df.groupby('product').agg(
    total_units=('units_sold', 'sum'),
    avg_units_per_day=('units_sold', 'mean'),
    total_revenue=('revenue', 'sum'),
    max_daily_units=('units_sold', 'max')
).sort_values('total_revenue', ascending=False)

agg_summary

## 11. Custom column with `apply`

In [ ]:
def demand_level(units):
    if units >= 60:
        return 'High'
    elif units >= 30:
        return 'Medium'
    else:
        return 'Low'

df['demand_level'] = df['units_sold'].apply(demand_level)
df['demand_level'].value_counts()

In [ ]:
df['demand_level'].value_counts().plot(kind='bar', title='Demand level distribution', color='seagreen')
plt.ylabel('Number of days')
plt.show()